# Module 3 — NGS Pipeline with Claude Code (Alignment)

**SBML Lab Intern Training | KAIST**

---

This module covers the full NGS alignment pipeline — from raw SRA data to a sorted, indexed BAM, a GFF file, and a visualization of the ChIP-exo signal in MetaScope. You'll use Claude Code to generate the pipeline commands, but you must understand what each step does before running it.

**Lab context:** Reference genome is *E. coli* K-12 MG1655. The dataset is **single-end ChIP-exo reads** — each read comes from one end of the DNA fragment, not a pair. Aligner is `bowtie2`. Post-alignment processing uses `samtools`, followed by `makegff.py` to produce a GFF file for MetaScope visualization.

**Learning objectives:**
- Understand how `bowtie2` aligns short reads to a reference genome
- Understand why BAM files must be sorted and indexed before downstream use
- Practice using Claude Code's **plan mode** for multi-step pipelines
- Annotate and verify every flag in a generated command before running it

## Background: NGS and ChIP-exo

### What is next-generation sequencing (NGS)?

A sequencing machine reads millions of short DNA fragments (called **reads**) in parallel and reports their sequences along with quality scores for each base. A single ChIP-exo experiment produces 10–50 million reads, each 50–150 base pairs long. Raw reads are stored in **FASTQ** format — one file, millions of entries.

Reads alone don't tell you anything. You need to figure out *where* in the genome each read came from. That's what alignment does.

### What is ChIP-exo?

**ChIP-exo** is a technique that maps exactly where a DNA-binding protein (a transcription factor) attaches to the chromosome:

1. **ChIP (chromatin immunoprecipitation):** Protein-DNA complexes are chemically cross-linked and then broken into fragments. An antibody specific to your protein of interest pulls down only the fragments it was bound to.
2. **Exo (exonuclease treatment):** A DNA-chewing enzyme trims the pulled-down fragments from their ends, leaving only the sequence right at the binding boundary. This is what makes ChIP-*exo* more precise than regular ChIP-seq.
3. **Sequencing:** The remaining fragments are sequenced. The resulting reads cluster tightly around the exact binding site.

### Why does this lab do ChIP-exo?

This lab studies transcription factors (TFs) — proteins that bind specific DNA sequences to control nearby genes. ChIP-exo lets us answer: *"Where in the E. coli genome does TF X bind, and under what conditions?"*

The dataset you'll work with in Modules 3 and 4 is a FUR ChIP-exo experiment — mapping where the iron-sensing protein FUR binds across the entire *E. coli* chromosome.

### The pipeline in one sentence

`FASTQ reads` → **align to reference genome** → `SAM/BAM alignment file` → **sort and index** → **convert to GFF** → `visualize in MetaScope`

Each step in this pipeline is one section of this module.

---

## 0. File Formats — Know Before You Align

This pipeline produces and consumes several file formats. Use Claude Code to answer the questions below **before** starting the exercises.

| Format | Question to answer with Claude Code |
|--------|-------------------------------------|
| FASTQ  | What do the 4 lines per read represent? What does the quality score number mean? |
| FASTA  | How does it differ from FASTQ? When would you use one vs the other? |
| SAM    | How many columns are there? What does the FLAG field encode? |
| BAM    | Why is BAM preferred over SAM for storage and downstream tools? |
| GFF    | How many columns? What coordinate system (0-based or 1-based)? |

Write your answers below — one sentence per format.


> **Answer**
>
> *(Write your one-sentence answers here.)*
1. FASTQ: 네 개의 줄은 각각 시퀀스 식별자, 염기 서열, 구분자(+)이고 각 염기의 quality score를 의미한다.
2. FASTA: quality score 없이 헤더와 서열 정보만 포함하며 FASTQ가 생어시퀀싱 리드 저장에 쓰이는 반면 FASTA는 주로 참조 유전체를 저장할 때 사용한다.
3. SAM: 11개의 필수 column으로 구성되며 FLAG는 매핑 방향이나 페어링 정보 등의 속성을 비트별 정수(bitwise integer)로 인코딩하여 나타낸다.
4. BAM: SAM을 압축한 binary 포맷이므로 저장 공간을 적게 차지하며 후속 분석 툴이 데이터를 인덱싱하고 처리하는 속도를 빠르게 한다.
5. GFF: 탭으로 구분된 9개의 열로 구성되어 있으며 유전체 특징의 위치를 나타낼 때 1-based coordinate system을 사용한다.

## 1. How `bowtie2` Works

### What is alignment?

**Alignment** is the process of finding where each sequenced read comes from in the reference genome. Imagine you have 20 million short sentences (reads), each 75 words long, and you want to find where each one appears in a 4.6-million-word book (the *E. coli* genome). Alignment does that — at scale, in seconds.

The output is a file where every read is annotated with its genomic position: chromosome, start coordinate, strand, and a mapping quality score.

### Why do we need an index?

Searching the entire genome for every read would be impossibly slow. `bowtie2-build` pre-processes the reference genome into a compressed **index** — a data structure that allows near-instant lookup. Think of it like a book index: instead of reading every page to find "FUR binding site," you jump straight to the relevant pages.

You build the index once; then alignment against that genome uses it every time.

`bowtie2` aligns short reads to a reference genome using an index built with `bowtie2-build`. Use Claude Code to understand how the aligner works before you run it.

### Exercise 1 — Understand bowtie2 Before Running It

Use Claude Code to understand how `bowtie2` finds alignments and what the key flags control. Write a 3-sentence summary in your own words in the cell below.

> **Answer**
>
> *(Write your 3-sentence summary here.)*
bowtie2는 FM-index(Burrows-Wheeler Transform 기반) 알고리즘을 활용해 ChIP 실험을 통해 얻은 짧은 reads를 대장균 표준 유전체에 빠르게 정렬한다. Key flags는 -x 플래그로 대장균 reference에 index 주소록 경로를 지정하고 위의 데이터는 single-end 이므로 리드 파일을 -U 플래그로 입력한다. 또한 최종 정렬 결과물은 -S 플래그를 통해 표준 텍스트 포맷인 SAM 파일로 출력되며 향후 samtools 가공 및 메타스케이프 시각화 파이프라인의 기초 자료가 된다. 

## 2. `samtools` Internals

### Why sort a BAM file?

After alignment, reads are stored in the order they were sequenced — essentially random with respect to genomic position. Most downstream tools (genome browsers, variant callers, coverage calculators) need to jump to a specific genomic region quickly. A **coordinate-sorted BAM** organizes reads by position, so a tool looking at position 1,234,567 doesn't have to scan the whole file — it jumps directly there.

**You must sort before you index.** An index built on an unsorted file is meaningless because the file's structure doesn't match what the index promises.

### Why index a BAM file?

The `.bai` index file is a lookup table that maps genomic positions to byte offsets in the BAM file. It's what allows `samtools view` to extract reads from chromosome position X in milliseconds. Without it, every query would require reading the entire BAM from the start.

`samtools` converts, sorts, and indexes alignment files. Use Claude Code to understand the difference between SAM and BAM, and why the sort step must come before the index step.

### Exercise 2 — Sort Before Index

**Without asking Claude Code first:** write in the cell below why you must sort a BAM file before indexing it. Use your own reasoning.

Then verify your explanation with Claude Code.

> **Answer**
>
> *(Write your reasoning here before consulting Claude Code.)*
인덱싱을 하는 이유는 유전체 지도에서 그 유전자의 위치를 빠르게 탐색하기 위한 것인데, 비슷한 기능을 하는 유전자는 어차피 위치적으로 모여있을텐데, 이것을 sorting 하지 않고 indexing 하는 것은 의미가 없고 컴퓨터가 잘 수행할 수 없을 것 같다. 

-> 실제 클로드 코드로 verify 한 것: 오페론을 제외하면 비슷한 기능의 유전자가 위치적으로 모여있지 않다. 그 이유는 특정 구간의 리드를 찾을 때 파일 전체를 스캔하지 않고 바로 그 오프셋으로 점프하기 위해서이다. 매핑이 연속된 범위로 성립하려면 좌표가 비슷한 리드들이 파일 안에서도 물리적으로 인접해야 하고 그러려면 좌표순 정렬이 선행되어야 한다.  


## 3. Plan Mode — Reviewing Pipelines Before Running

Claude Code's **plan mode** lets you see every step Claude intends to take before a single command is executed.

**How to activate:** Press **Shift+Tab** before sending your prompt, cycling until the status line shows plan mode is on (a single press may land on a different mode). The interface switches to plan mode. Claude will show you the full sequence of actions — file reads, commands, edits — and wait for your approval.

**Why this matters for pipelines:**  
A multi-step NGS pipeline touches raw data, builds index files, and writes intermediate outputs. A mistake at step 2 (wrong index path) silently produces a valid-looking but incorrect BAM. Reviewing the plan lets you:
- Catch wrong paths or flag values before they cause silent errors
- Understand every command before it runs
- Modify individual steps (e.g., add `--no-unal`, increase `-p` threads) without re-generating the whole plan

**Rule for this module:** Any time you ask Claude Code to generate a multi-step pipeline, you must use plan mode and review it before approving.

### Exercise 3 — Plan Mode Pipeline Review

Use plan mode (Shift+Tab twice) to generate the full pipeline:
```
FASTQ → bowtie2 align → samtools sort → samtools index → makegff.py
```
Review every proposed step before approving. Identify at least one flag or parameter you would change or verify, and explain why.

> **Answer**
>
> *(Write the flag/parameter you identified, your change, and your reasoning here.)*

claude가 짠 pipeline 코드:

```bash
SRR=SRR_accession
REF=data/reference/NC_000913.3.fasta
IDX=data/reference/NC_000913.3
READS=data/reference/${SRR}_1.fastq
SAMPLE=fur_chipexo

# Step 1: Build bowtie2 index from reference FASTA
bowtie2-build ${REF} ${IDX}

# Step 2: Align single-end reads with bowtie2
# -U: single-end reads input ; -p 4: use 4 threads ; --no-unal: drop unaligned reads from output
bowtie2 -x ${IDX} -U ${READS} -p 4 --no-unal -S notebooks/${SAMPLE}.sam

# Step 3: Convert SAM to BAM (samtools view)
samtools view -bS notebooks/${SAMPLE}.sam > notebooks/${SAMPLE}.bam

# Step 4: Sort BAM by coordinate (samtools sort)
samtools sort -@ 4 notebooks/${SAMPLE}.bam -o notebooks/${SAMPLE}_sorted.bam

# Step 5: Index sorted BAM (samtools index)
samtools index notebooks/${SAMPLE}_sorted.bam

# Step 6: Run your makegff.py to generate GFF for MetaScope
python notebooks/makegff.py notebooks/${SAMPLE}_sorted.bam notebooks/${SAMPLE}_chipexo.gff
```

- 'bowtie2' 단계에서 '-p 4' 와 'samtools sort' 단계에서 '-@ 4' 는 둘다 스레드 개수를 지정하는 flag이다. claude가 짜준 계획안에는 병렬처리를 할 때 4개의 스레드를 사용하도록 하였지만, CPU나 RAM 스펙이 충분하다면 스레드 값을 8로 바꾸어도 될 것 같다. 

## 4. Downloading Your Input Data

The pipeline needs **two** inputs, both in `data/reference/`: the sequencing **reads** and the **reference genome** they align to. You'll download both with Claude Code in this section.

### Exercise 4 — Reads: single-end ChIP-exo (`fastq-dump`)

The reads come from the FUR ChIP-exo study you'll work with in Module 4: Seo et al. 2014, GEO series **GSE54901**. Use Claude Code to find and download an appropriate single-end ChIP-exo sample into `data/reference/`.

In [8]:
%%bash
# Exercise 4 — Use Claude Code to download the ChIP-exo reads from GSE54901.
# Paste the final command(s) here and annotate each flag with a comment.
# 1. 데이터를 저장할 디렉토리를 생성. (cwd가 notebooks/ 이므로 ../data/reference/ = repo top-level data/reference/)
mkdir -p ../data/reference/

# 2. NCBI 정식 주소에서 SRA 파일을 다운로드.
curl --insecure -L -o ../data/reference/SRR1168121.sra "https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR1168121/SRR1168121"

# 3. 로컬에 받아진 파일로 fastq-dump를 수행. 
# # fastq-dump: NCBI SRA(Sequence Read Archive) 에서 원본 압축 서열(.sra)를 다운로드하고, 실제 Alignment에 사용할 수 있는 표준 텍스트 서열인 FASTQ(.fastq.gz) 파일로 변환 및 추출해 주는 NCBI SRA Toolkit의 핵심 명령행 도구(Command-line tool)이다.
fastq-dump \
  --outdir ../data/reference/ \
  --gzip \
  ../data/reference/SRR1168121.sra

# 4. 파일명 정리
mv ../data/reference/SRR1168121.sra.fastq.gz ../data/reference/SRR1168121.fastq.gz 2>/dev/null || true

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
107 42.6M   77 33.0M    0     0  7568k      0  0:00:05  0:00:04  0:00:01 7566k --:--:-- --:--:-- --:--:--     0    0   144k      0  0:05:03  0:00:01  0:05:02  144k   0  0:00:18  0:00:02  0:00:16 2410k5968k0 42.6M  100 42.6M    0     0  8473k      0  0:00:05  0:00:05 --:--:-- 9413k


Read 2631506 spots for ../data/reference/SRR1168121.sra
Written 2631506 spots for ../data/reference/SRR1168121.sra


### Exercise 4b — Reference Genome: *E. coli* K-12 MG1655 (`NC_000913.3`)

Alignment needs the genome the reads came from. Use Claude Code to download the *E. coli* K-12 MG1655 reference genome — accession **NC_000913.3** — as a FASTA into `data/reference/`, saved as `NC_000913.3.fasta`. (Hint: the `entrez-direct` tools in this container can fetch sequences by accession from NCBI.)

This is the file `bowtie2-build` turns into the alignment index in Exercise 7 — without it the pipeline's first step fails.

In [10]:
%%bash
# Exercise 4b — Use Claude Code to download the reference genome NC_000913.3 as FASTA
#               into data/reference/NC_000913.3.fasta.
# Paste the final command here and annotate what it does.
# (cwd가 notebooks/ 이므로 ../data/reference/ = repo top-level data/reference/)

efetch -db nuccore -id NC_000913.3 -format fasta > ../data/reference/NC_000913.3.fasta
# -db nuccore: NCBI's nucleotide database, where full chromosome/genome records live
# -id NC_000913.3: RefSeq accession for the E. coli K-12 MG1655 chromosome
# -format fasta: sequence-only FASTA (not the GenBank flat file, which also has annotations)
ls -lh ../data/reference/NC_000913.3.fasta

curl: (56) OpenSSL SSL_read: OpenSSL/3.5.6: error:0A000126:SSL routines::unexpected eof while reading, errno 0
 ERROR:  curl command failed ( Sun Jul 19 14:23:52 UTC 2026 ) with: 56
-X POST https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi -d db=nuccore&id=NC_000913.3&rettype=fasta&retmode=text&tool=edirect&edirect=25.3&edirect_os=Linux&email=vscode%40codespaces-4a290c
HTTP/1.0 200 OK


-rw-rw-rw- 1 vscode vscode 2.7M Jul 19 14:24 ../data/reference/NC_000913.3.fasta


## 5. SAM FLAG Values

Every SAM record has a FLAG field in column 2 — a bitwise integer encoding alignment properties. Use Claude Code to understand what FLAG values mean and how to filter by them with `samtools view`.

### Exercise 5 — Decode an Unknown FLAG

> Look up what a specific FLAG value means using Claude Code. Choose a value you have **not** looked up yet (e.g., 2048, 256, 1024).
>
> Write:
> 1. The FLAG value you chose
> 2. What it means (decoded bits)
> 3. How you would verify that a real SAM record carries this flag (e.g., with `samtools view -f` or `samtools flags`)

> **Answer**
>
> *(Write your FLAG value, its meaning, and verification method here.)*
1. 256 이라는 FLAG는 Reference Genome 상에 리드가 매핑되었을 때, 최적의 위치가 아닌 곳에 정렬되었음을 의미한다. 
2. read 서열이 너무 흔하면 reference에 다중 매핑이 된다. 이때 매핑 프로그램이 내부 점수 계산을 통해 주 매핑과(primary alignment) 차선책 매핑(secondary alignment)을 구분하고 차선책 매핑에는 FLAG 256이 적히게 된다.
3. samtools view -f 256 input.sam: SAM 파일에서 samtools view 명령어에 -f 256 옵션을 사용하면 전체 정렬 데이터 중에서 오직 차선책 매핑 플래그가(256) 포함된 리드들만 필터링 해 화면에 출력할 수 있다.

## 6. Writing `makegff.py`

Your sorted BAM holds every aligned read in a binary format bioinformatics tools understand — but **MetaScope**, the lab's genome browser, reads **GFF**, not BAM. `makegff.py` is the converter, and it's the last step of the pipeline.

But it isn't a dumb format swap — *how* you turn reads into GFF rows decides whether you can actually see a binding site. Two things drive that:

**MetaScope draws each GFF feature at a height set by the score column.** So the score is not decoration — it's the signal. If you want a binding site to rise as a **peak**, the score has to carry *how much evidence* is at each position.

**ChIP-exo puts the signal at the read's 5′ end.** A lambda exonuclease chews each bound DNA fragment 5′→3′ until it stops at the protein–DNA crosslink. So the **5′ end of every read marks a crosslink boundary** — that's the informative position, not the whole read body:
- for a **`+`-strand** read, the 5′ end is its **start**;
- for a **`−`-strand** read, the 5′ end is its **end**.

Many fragments from the same binding event pile their 5′ ends at the same spot. So the right GFF encodes, **per strand**, *how many reads share each 5′-end position* — and writes that **count into the score column**. That's what makes a binding site show up as a tall, sharp peak. (The crosslink is flanked by a `+`-strand peak and a `−`-strand peak; the true site sits **between** them — which is why strands are kept separate.)

This script is **not pre-provided** — you'll write it with Claude Code. Consult `lab-context.md` for the exact GFF column structure this lab uses.

### Exercise 6 — Write `makegff.py` (5′-end pileup)

Use Claude Code to write a Python script `makegff.py` that turns a sorted BAM into a GFF MetaScope can display as a ChIP-exo signal:

1. Takes a sorted BAM as input, writes a GFF file.
2. For each aligned read, take its **5′ end** (`start` if `+`, `end` if `−`).
3. **Count reads per (strand, 5′-end position)** and write **one GFF row per occupied position**, with the **count in the score column** so pile-ups become peaks.
4. Keep `+` and `−` strands **separate** (this is the lab script's `--separate_strand` behavior).
5. *(optional)* add a `--log_scale` mode that writes `log2(count+1)` instead of the raw count, so a few very strong sites don't visually flatten the weaker real ones.

**Output format** (one row per occupied 5′-end position; coordinates **1-based**):
```
NC_000913.3	makegff	fiveprime	12345	12345	7	+	.	depth=7
```
Columns: seqname, source (`makegff`), feature (`fiveprime`), start, end (= same position), **score = read count**, strand, `.`, attribute.

Test it on your aligned BAM before running the full pipeline in Exercise 7.

> **Explain it:** after your code runs, write 1–2 sentences on *why counting 5′ ends per strand (rather than one row per read) is what makes a binding site visible*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

Explain it에 대한 답: read 전체 대신, strand별로 5' 의 위치를 카운트하는 것이 binding site를 명확히 하는 이유는 다음과 같다. Binding site 주변은 효소 절단이 특정 위치에서 집중적으로 일어나는데 리드 전체를 확인하면 50~150bp에 달하는 길이에 신호가 넓게 퍼져버리지만, 5'end 위치만 찍어서 카운트하면 뾰족한 peak를 형성시켜 식별이 잘 되기 때문이다. (Exonuclease는 5' 끝에서부터 출발해 갉아먹다가 단백질을 만나면 걸려서 멈춘다.)

In [ ]:
%%writefile makegff.py # 셀 매직 명령어. 셀 전체에 적용이 돼고 첫번째 줄에 작성해야 한다. makegff.py라는 파일을 생성 후 저장한다. 
import argparse 
import math # 수학 함수를 제공하는 라이브러리. 
from collections import Counter # dict와 유사한 자료구조. key는 중복 불가, value는 중복 가능.

import pysam # SAM/BAM 파일을 다루는 라이브러리.


def five_prime_counts(bam_path): 
    # 정렬된 BAM에서 매핑된 read마다 5' end 좌표(1-based)를 strand별로 세어 반환.
    counts = Counter()
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam.fetch():
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue

            strand = "-" if read.is_reverse else "+"
            # 5' end: '+' strand는 정렬 시작점, '-' strand는 정렬 끝점.
            # reference_start는 0-based라 +1로 1-based 보정, reference_end는 이미 그 위치와 같다.
            position = read.reference_end if read.is_reverse else read.reference_start + 1

            # annotation GFF는 'NC_000913'을 쓰는데, BAM의 reference_name은 fasta 헤더 그대로인 'NC_000913.3'이라 MetaScope에서 서로 다른 chromosome으로 인식되므로. 버전 suffix를 잘라 맞춘다.
            ref_name = read.reference_name.split(".")[0]

            counts[(ref_name, strand, position)] += 1

    return counts


def write_gff(counts, gff_path, log_scale=False):
    # (seqname, strand, position)별 count를 MetaScope용 GFF 한 줄로 기록.
    with open(gff_path, "w") as gff: # 출력할 GFF 파일을 쓰기 모드("w")로 염.
        for (seqname, strand, position), count in sorted(
            counts.items(), key=lambda kv: (kv[0][0], kv[0][2], kv[0][1])  # 정렬 기준: seqname(염색체 이름), position(위치), strand(가닥)
        ):
            score = round(math.log2(count + 1), 3) if log_scale else count 
            gff.write(
                f"{seqname}\tmakegff\tfiveprime\t{position}\t{position}\t{score}\t{strand}\t.\tdepth={count}\n"
            )


def main():
    parser = argparse.ArgumentParser(  # 터미널 명령어를 파싱하기 위한 ArgumentParser 객체를 생성. 터미널에서 사용자가 입력한 인자들을 파이썬 변수로 변환해주기 위한 도구.
        description="Convert a sorted BAM into a 5'-end pileup GFF for MetaScope."
    )
    parser.add_argument("bam", help="sorted, indexed BAM file") # 1번째 필수 입력인자
    parser.add_argument("gff", help="output GFF path") # 2번째 필수 입력인자
    parser.add_argument(
        "--log_scale", # log_scale 플래그를 추가해 입력 시 점수를 로그 스케일로 계산하도록 하는 추가 옵션. 
        action="store_true",
        help="score = log2(count+1) instead of the raw read count",
    )
    args = parser.parse_args()

    counts = five_prime_counts(args.bam)
    write_gff(counts, args.gff, log_scale=args.log_scale)


if __name__ == "__main__":
    main()

## 7. Running the Full Pipeline

### Exercise 7 — Run the Full Pipeline

By now `data/reference/` should hold both inputs from Section 4: the reference genome (`NC_000913.3.fasta`, Exercise 4b) and the ChIP-exo reads (Exercise 4). Use plan mode to generate the full pipeline, then fill in each command below after reviewing the plan.

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

Explain it에 대한 답:
[파이프라인; How it works] 
FASTQ -> Bowtie2(매핑) -> SAM -> BAM -> Sort -> Index -> makegff.py

[Why Claude chose that approach]
    메모리/속도 효울성 측면: 대용량 유전체 데이터를 bianry 파일(BAM)로 변환 후 sort하였고 Thread 수를 환경에 맞게 지정해 리소스 낭비를 막을 수 있기 때문이다. 

    데이터 정밀도 측면: 전체 read의 넓은 coverage 대신 ChIP-exo의 핵심인 5' end 만을 집계하여 시각화(MetaScope)에 맞는 GFF 표준 형식으로 변환할 수 있기 때문이다. 

In [ ]:
%%bash
set -e

# Step 1: Build bowtie2 index from reference FASTA
bowtie2-build ../data/reference/NC_000913.3.fasta ../data/reference/NC_000913.3

# Step 2: Align single-end reads with bowtie2
bowtie2 -x ../data/reference/NC_000913.3 -U ../data/reference/SRR1168121.fastq.gz -p 2 -S fur_chipexo.sam
# -x: 정렬에 쓸 index prefix
# -U: single-end reads 입력
# -p 2: 정렬에 쓸 스레드 수. nproc이 2라서 2로 맞춤 - 그 이상은 오버섭스크립션이라 느려짐
# -S: SAM(텍스트) 형식으로 출력할 경로

# Step 3: Convert SAM to BAM (samtools view)
samtools view -bS fur_chipexo.sam > fur_chipexo.bam
# -b: 출력을 BAM(바이너리) 형식으로 만듦
# -S: 입력이 SAM 형식임을 명시 (최신 버전 samtools는 자동 감지 가능하므로 생략 가능)

# Step 4: Sort BAM by coordinate (samtools sort)
samtools sort -@ 2 -o fur_chipexo_sorted.bam fur_chipexo.bam
# -@ 2: 정렬에 쓸 스레드 수. nproc이 2라서 2로 맞춤
# -o: 출력 파일(BAM) 경로 지정

# Step 5: Index sorted BAM (samtools index)
samtools index fur_chipexo_sorted.bam

# Step 6: Run your makegff.py to generate GFF for MetaScope
python makegff.py fur_chipexo_sorted.bam fur_chipexo_chipexo.gff

Settings:
  Output files: "../data/reference/NC_000913.3.*.bt2"
  Line rate: 6 (line is 64 bytes)
  Lines per side: 1 (side is 64 bytes)
  Offset rate: 4 (one in 16)
  FTable chars: 10
  Strings: unpacked
  Max bucket size: default
  Max bucket size, sqrt multiplier: default
  Max bucket size, len divisor: 4
  Difference-cover sample period: 1024
  Endianness: little
  Actual local endianness: little
  Sanity checking: disabled
  Assertions: disabled
  Random seed: 0
  Sizeofs: void*:8, int:4, long:8, size_t:8
Input files DNA, FASTA:
  ../data/reference/NC_000913.3.fasta
Reading reference sizes


Building a SMALL index


  Time reading reference sizes: 00:00:00
Calculating joined length
Writing header
Reserving space for joined string
Joining reference sequences
  Time to join reference sequences: 00:00:00
bmax according to bmaxDivN setting: 1160413
Using parameters --bmax 870310 --dcv 1024
  Doing ahead-of-time memory usage test
  Passed!  Constructing with these parameters: --bmax 870310 --dcv 1024
Constructing suffix-array element generator
Building DifferenceCoverSample
  Building sPrime
  Building sPrimeOrder
  V-Sorting samples
  V-Sorting samples time: 00:00:00
  Allocating rank array
  Ranking v-sort output
  Ranking v-sort output time: 00:00:00
  Invoking Larsson-Sadakane on ranks
  Invoking Larsson-Sadakane on ranks time: 00:00:00
  Sanity-checking and returning
Building samples
Reserving space for 12 sample suffixes
Generating random suffixes
QSorting 12 sample offsets, eliminating duplicates
QSorting sample offsets, eliminating duplicates time: 00:00:00
Multikey QSorting 12 samples
  (Using

Renaming ../data/reference/NC_000913.3.3.bt2.tmp to ../data/reference/NC_000913.3.3.bt2
Renaming ../data/reference/NC_000913.3.4.bt2.tmp to ../data/reference/NC_000913.3.4.bt2
Renaming ../data/reference/NC_000913.3.1.bt2.tmp to ../data/reference/NC_000913.3.1.bt2
Renaming ../data/reference/NC_000913.3.2.bt2.tmp to ../data/reference/NC_000913.3.2.bt2
Renaming ../data/reference/NC_000913.3.rev.1.bt2.tmp to ../data/reference/NC_000913.3.rev.1.bt2
Renaming ../data/reference/NC_000913.3.rev.2.bt2.tmp to ../data/reference/NC_000913.3.rev.2.bt2
[WARNING] Failed to launch x86-64-v3 version, staying with default
[WARNING] Failed to launch x86-64-v3 version, staying with default
2631506 reads; of these:
  2631506 (100.00%) were unpaired; of these:
    20310 (0.77%) aligned 0 times
    2533631 (96.28%) aligned exactly 1 time
    77565 (2.95%) aligned >1 times
99.23% overall alignment rate
[bam_sort_core] merging from 0 files and 2 in-memory blocks...


## 8. Visualizing ChIP-exo in MetaScope

You now have a GFF where each 5′-end pile-up is scored by read count. Numbers in a file don't tell you where FUR binds — you need to *see* the peaks along the genome next to the genes they sit near. That's what a **genome browser** does.

### What is a genome browser?

A genome browser draws the genome as a horizontal axis (position 1 → 4.6 Mb for *E. coli*) and stacks your data on top as **tracks**. Each track is one GFF file. Load the gene annotation as one track and your ChIP-exo pile-up as another, and you can read directly off the screen: *this peak sits right in front of that gene.*

### MetaScope — the lab's browser

**MetaScope** is the SBML lab's own genome browser. It reads **GFF files** (exactly what `makegff.py` produced), drawing each feature at a height set by its **score**, and lets you overlay tracks, zoom, jump to a position, search by gene name, and export a publication-quality figure.

- **It is a desktop application** — it runs on your own computer (Windows or macOS), **not** inside the Codespace. The Codespace is where you *make* the GFF; MetaScope is where you *look* at it.
- **Download and install it from the lab homepage** ([sbml-lab.ai](https://sbml-lab.ai)). Pick the build for your operating system.
- The workflow is: **produce the GFF in the Codespace → download the GFF to your machine → open it in MetaScope → cross-reference with the annotation → export a figure.**

### The two tracks you'll load

| Track | File | What it shows |
|-------|------|---------------|
| Gene annotation | `ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (from Module 2) | Where every gene is |
| ChIP-exo 5′-end pile-up | your `makegff.py` output (Exercise 6/7) | Read-count peaks where FUR-bound fragments piled up |

Overlaid, a tall peak of ChIP-exo 5′ ends just upstream of a gene is a candidate **FUR binding site** regulating that gene.

### Exercise 8 — What Should a Binding Site Look Like? (write before you look)

**Before installing anything**, write your prediction in the cell below — reason it out yourself:

1. ChIP-exo enriches DNA fragments that were bound by the FUR protein. When those reads are aligned and drawn as a track, **what shape** would you expect at a real binding site versus a random stretch of genome?
2. FUR is a **repressor** of iron-uptake genes. Relative to a gene it controls, **where** along the genome would you expect its binding site to sit — inside the gene, or somewhere else?

Then verify your reasoning with Claude Code (`/explain ChIP-exo peak shape` or ask directly).

> **Answer (instructor key)**
>
> 1. Shape:
 - FUR 결합 부위는 주변 노이즈(baseline)보다 훨씬 높고 뾰족하게 솟아오른 국소적인 Peak (sharp, localized peak) 형태일 것이다.
 - ChIP-exo 기법은 단일 염기 수준의 극도로 높은 해상도를 제공하기 때문에, 신호가 넓게 퍼지지 않고 수십 bp 수준의 매우 좁은 범위에 집중될 것이다.
 - 특히 strand를 분리하여 카운트하므로 실제 단백질이 결합한 지점(crosslink)을 기준으로 좌우 양옆에 정방향 가닥 Peak와 역방향 가닥 Peak가 대칭으로 감싸는 형태이며, 진짜 결합 위치는 이 두 Peak의 사이일 것이다.
>


> 2. Location:
- FUR 단백질은 철 이온 흡수 유전자의 전사를 물리적으로 방해하는 repressor 역할을 할 것이다.
- 따라서 유전자 내부(coding body)가 아닌, RNA 중합효소의 결합을 직접 차단할 수 있는 promoter/operator에 결합할 것이다.
- 최종적으로 Peak 신호는 조절하려는 타겟 유전자 시작점(start codon)의 바로 앞쪽인 5' 영역에 존재할 것이다.

특히 배경 지식과 힌트를 참고하면 대장균의 주요 철 gain system인 fepA/fes 영역(609–613 kb)이나 fhuA 영역(160–170 kb) 등이 대표적인 chIP-exo의 타겟일 것이다. 따라서 실제 MetaScope 상에서 데이터를 확인할 때는 해당 유전자들의 시작점 바로 앞쪽인 5′ 상류 영역 부근에서 뚜렷한 국소적인 피크가 관찰될 것이다.  

### Exercise 9 — Get Your GFFs Out of the Codespace

MetaScope runs on your machine, so both tracks need to be on your machine. Run the cell below to confirm both GFFs exist and see their sizes, then download them.

**To download a file from a Codespace:** in the VS Code file explorer (left panel), right-click the file → **Download…**, and save it somewhere you'll find it (e.g. your Desktop).

Download **both**:
- `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (the annotation)
- your ChIP-exo GFF from Exercise 6/7 (e.g. `notebooks/chipexo.gff` — use whatever name you gave it)

In [ ]:
%%bash
# Confirm both tracks exist and are non-empty before you download them.
# Adjust the ChIP-exo filename to whatever you named your makegff.py output.

echo "== Annotation track =="
ls -lh data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
head -2 data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff

echo
echo "== ChIP-exo track (edit the filename if yours differs) =="
CHIP_GFF=notebooks/chipexo.gff
ls -lh "$CHIP_GFF" && head -2 "$CHIP_GFF" && echo "rows: $(wc -l < "$CHIP_GFF")"

### Exercise 10 — Load, Locate, and Identify

Open MetaScope on your machine and:

1. **Open both GFFs** (File → Open, or `Ctrl+O` / `Cmd+O`): the annotation *and* your ChIP-exo 5′-end pile-up track.
2. **Find a binding site.** Look for a location where the ChIP-exo track shows a clear, tall **peak** of read counts. Use `Ctrl/Cmd+G` to jump to a position and scroll/zoom to explore. (Iron-uptake regions worth checking: **160–170 kb** near *fhuA*, **609–613 kb** at the *fepA*/*fes* pair, **624–629 kb** at the enterobactin *entCEBA* genes, **1,733 kb** near *sodB*.) If you kept strands separate, the true crosslink sits **between** the flanking `+` and `−` peaks.
3. **Identify the nearest gene.** Hover over a gene feature for its tooltip (position, strand, name), or `Ctrl/Cmd+F` to search the annotation by name. Read off which gene the peak sits **upstream of**.

> **Heads-up — a real gotcha.** MetaScope groups tracks by their **chromosome ID** (column 1 of the GFF). The annotation uses `NC_000913`, but `makegff.py` writes `NC_000913.3` (the accession of the reference FASTA). If your two tracks land on **separate chromosome tabs** and won't line up, that's why. Use `/debug` to reason it out, then fix it (make the two column-1 values match) and reload. Worth verifying rather than assuming.

4. **Export your figure** (`Ctrl+Shift+E` → PNG, 300 dpi) as `module3_chipexo_metascope.png`. This is your Module 3 deliverable.

In the answer cell, record: the genomic position of the site, the nearest gene, and whether the binding site is upstream of it (as you predicted in Exercise 8).

> **Answer**
> - Attach `module3_chipexo_metascope.png` (your exported MetaScope figure) to your submission.

질문 답: 
[1번째로 찾은 protein binding site]
* Binding-site position (approx.): ~167,457 bp
* Nearest gene / locus_tag: fhuA / b0150
* Is the site upstream of the gene? Does it match your Exercise 8 prediction?: 
  관찰된 피크는 fhuA 유전자의 시작점 바로 앞쪽인 5′ 상류 영역(upstream)에 정확히 위치한다. 이는 FUR protein이 repressor로서 프로모터 영역에 결합해 전사를 차단할 것이라 excercise 8에서 예측한 것과 일치한다.

[2번째로 찾은 protein binding site]
* Binding-site position (approx.): ~624,860 bp
* Nearest gene / locus_tag: entE / b0593 (또는 entC / b0592)
* Is the site upstream of the gene? Does it match your Exercise 8 prediction?: 
  MetaScope 상에서는 초록색 박스 내부 한가운데에 피크가 존재하는 것처럼 보이지만 이는 entC-entE-entB-entA 유전자들이 'entCEBA 오페론' 으로 구성되어 있기 때문이다. 따라서 오페론 내부에서 다음 유전자(entE 등)가 시작되기 전의 내부 조절 영역(Internal upstream/promoter)에 결합한 것으로 해석할 수 있다. 결과적으로 이것 역시 FUR 단백질이 유전자 발현을 제어하는 조절 부위에 결합한다는 Exercise 8의 예측과 일치한다.

### Exercise 11 — Cross-Check the Biology

You found a gene next to a FUR binding site. Now confirm it makes biological sense — don't take the browser (or Claude) at face value.

Ask Claude Code: *is this gene a known FUR target, and what does it do?* A correct result should be an **iron-acquisition or iron-storage gene** (siderophore transport, enterobactin biosynthesis, iron-superoxide dismutase, etc.) — FUR represses these under iron-replete conditions.

If the gene has nothing to do with iron, that's a signal to re-check: did you pick the right peak? Is it really the *nearest* gene? Are the two tracks on the same chromosome tab? Write what you found.

> **Answer**
>
> *(What does the gene do? Is it a known FUR target? Did anything not line up, and how did you resolve it?)*

1번 유전자(fhuA) 검증:
본 과제에서 발견한 fhuA gene은 외막의 TonB-dependent 수용체 단백질로서 철-시데로포어 복합체(ferrichrome)를 세포 내부로 수송하는 대표적인 철분 흡수 유전자이다. 이는 fhuACDB 오페론의 첫 번째 유전자로 철이 풍부한 환경에서 FUR 단백질에 의해 전사가 억제되는 FUR 타겟이다. 피크 위치가 metascope의 유전자 박스의 앞에 정렬되어 이것이 repressor로서 작용함을 검증할 수 있었다.

3번 유전자(entCEBA 오페론) 검증:
본 과제에서 발견한 entE / entC gene은 대장균의 핵심 시데로포어인 enterobactin의 생합성을 담당하는 효소들을 인코딩한다. 이는 entCEBA 오페론에 속하며 세포 내 철분이 충분할 때 FUR 단백질에 의해 억제되는 대표적인 철분 흡수 유전자 군집이다. 오페론 내부의 정렬된 것을 확인한 결과 예측한 결합 자리와 하위 유전자 조절 부위가 일치하여 타당한 결과임을 검증할 수 있었다.

## End of Session

Before closing:

1. Make sure your pipeline cell runs end-to-end without errors.
2. Verify that the output BAM is coordinate-sorted: `samtools view -H output.bam | grep SO`
3. Verify the index exists: `ls -lh output.bam.bai`
4. Verify your GFF output has content: `wc -l output.gff`
5. In MetaScope, load your GFF (Section 8) and export the figure `module3_chipexo_metascope.png`.
6. Run `/log` in Claude Code to save a session log.

---

**Run `/log` before closing.**

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module3): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
